# $${\color{orange}Preprocessing-EDA}$$

### Preprocessing (Data Mining / Null Values / Drop Columns / General Cleaning):

In [ ]:
%pip install plotly tensorflow flask matplotlib 
%pip install pandas

In [ ]:
# import io
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

In [3]:
train_gene = pd.read_csv(r"./train.csv")
test_gene = pd.read_csv(r"./test.csv")

In [ ]:
train_gene.shape

In [ ]:
test_gene.shape

In [ ]:
train_gene.head()

In [ ]:
train_gene.info()

In [ ]:
train_gene.describe()

In [ ]:
train_gene.describe(include='O')

In [ ]:
train_gene.duplicated().sum()

In [11]:
# List of columns to drop
columns_to_drop = [
    "Patient Id", "Patient First Name", "Family Name", "Father's name", "Location of Institute",
    "Genes in mother's side", "Institute Name", "Test 1", "Test 2", "Test 3", "Test 4",
    "Test 5", "Symptom 1", "Symptom 2", "Parental consent", "Inherited from father",
    "Place of birth", "Symptom 3", "Symptom 4", "Symptom 5"]

# Drop columns from both train_gene and test_gene
for df in [train_gene, test_gene]:
    df.drop(columns=columns_to_drop, inplace=True)


In [12]:
# replace unknown values by nan :
for df in [train_gene, test_gene]:
    df.replace(['-99', '-'], np.nan, inplace=True)

In [ ]:
# Checking Null Values :
train_gene.isna().mean()

In [14]:
# Treating Null Values :
catego_cols = train_gene.select_dtypes(include="O")
for col in catego_cols:
    train_gene[col] = train_gene[col].fillna(train_gene[col].mode().iloc[0])
numri_cols = train_gene.select_dtypes(include=['float', 'int'])
for col in numri_cols:
    train_gene[col] = train_gene[col].fillna(train_gene[col].median())

In [ ]:
# Replacing Spaces with Underscore :
# Either took substance or not (making categories yes or no):
for df in [train_gene, test_gene]:
    df.columns = df.columns.str.replace(" ", "_")
    df['H/O_substance_abuse'].replace('Not applicable', 'No', inplace=True)


In [16]:
# Define the renaming dictionary
rename_dict = {
    'Blood_cell_count_(mcL)': 'Blood_cell_count',
    "Respiratory_Rate_(breaths/min)": "Respiratory_Rate",
    'Autopsy_shows_birth_defect_(if_applicable)': 'Autopsy_shows_birth_defect',
    'Folic_acid_details_(peri-conceptional)': 'Folic_acid',
    'No._of_previous_abortion': 'Previous_abortion',
    "Mother's_age": "Mother_age",
    "Father's_age": "Father_age",
    'H/O_radiation_exposure_(x-ray)': 'radiation_exposure_x_ray',
    'H/O_serious_maternal_illness': 'serious_maternal_illness',
    'Heart_Rate_(rates/min': 'Heart_Rate',
    'White_Blood_cell_count_(thousand_per_microliter)': 'White_Blood_cell_count_thousand_per_microliter'
}

# Rename columns in both train_gene and test_gene
for df in [train_gene, test_gene]:
    df.rename(columns=rename_dict, inplace=True)

In [17]:
columns_to_convert = ['Patient_Age', 'Father_age', 'Mother_age']
# Convert each column to numeric with coercion for errors
for column in columns_to_convert:
    train_gene[column] = pd.to_numeric(train_gene[column], errors='coerce')
    test_gene[column] = pd.to_numeric(test_gene[column], errors='coerce')

In [18]:
disorder_mapping = {
    'Mitochondrial genetic inheritance disorders': 'Mitochondrial',
    'Multifactorial genetic inheritance disorders': 'Multifactorial',
    'Single-gene inheritance diseases': 'Single_gene'
}

# Apply the mapping to train_gene :
train_gene['Genetic_Disorder'] = train_gene['Genetic_Disorder'].replace(disorder_mapping)

In [ ]:
for df in [train_gene, test_gene]:
    df['Birth_asphyxia'].replace('Not available', 'No record', inplace=True)

### EDA:

#### Univariate Analysis :

In [20]:
#Categorical & Numerical columns :
cat_cols = train_gene.select_dtypes(include = "O").columns.to_list()
num_cols = train_gene.select_dtypes(include=['float', 'int']).columns.to_list()

In [ ]:
# Quick look on categorical columns :
import math
from matplotlib import pyplot as plt
# Determine the optimal number of rows and columns
n_cols = math.ceil(math.sqrt(len(cat_cols)))
n_rows = math.ceil(len(cat_cols) / n_cols)
# Create the figure with an appropriate size
fig = plt.figure(figsize=(n_cols * 4, n_rows * 4))
# Plot each categorical column as a bar plot
for i, col in enumerate(cat_cols):
    ax = fig.add_subplot(n_rows, n_cols, i + 1)
    train_gene[col].value_counts().plot(kind="bar", ax=ax, title=col, rot=40)
# Adjust layout
fig.tight_layout()
plt.show()

In [ ]:
# Quick look on Numerical columns :
# Select numerical columns
numerical_gene = train_gene.select_dtypes(include=['float', 'int'])
# Create boxplot
plt.figure(figsize=(10, 5))
numerical_gene.boxplot()
plt.xticks(rotation=45, ha='right')
plt.title('Boxplot of Numerical Columns')
plt.show()

#### Bivariate & Multivariate Analysis :

##### Disorder Classes With Sub-classes :

In [ ]:
disease = train_gene.groupby('Genetic_Disorder')['Disorder_Subclass'].value_counts(normalize=True)
disease = disease.unstack()
fig, ax = plt.subplots(figsize=(7, 8))  # Adjust the figsize (width, height) as needed
disease.plot(kind='bar', stacked=True, color=sns.color_palette("Set2", 9), ax=ax)
plt.xticks(rotation=45)
plt.show()

##### Correlation between Numerical Values :

In [ ]:
# Calculate correlation matrix for numerical columns
numerical_gene.corr()

In [ ]:
corr_map = sns.heatmap(numerical_gene.corr() ,  cmap='coolwarm' ,  annot=True)
corr_map

##### Maternal gene effect on all Disorders :

In [ ]:
# Maternal Gene for each Disorder_Subclass:
maternal = sns.catplot(
    data = train_gene, y="Disorder_Subclass", hue="Maternal_gene", kind="count",
    palette="pastel", edgecolor=".6",)
maternal

In [27]:
# Maternal gene is of higher effect than that of Paternal gene.

##### Is there a relation between Gender and the presence of a Genetic Disorder?

In [ ]:
fig2 = sns.countplot(x='Gender', hue='Disorder_Subclass', data=train_gene)
fig2

In [29]:
# Leigh Syndrome is of very percentage in ambiguous gender.

In [ ]:
pl = train_gene['Birth_defects'].value_counts().reset_index()
pl

##### Does the Birth_asphyxia status relate to the occurrence of Birth_defects?

In [ ]:
fig3 = sns.countplot(x='Birth_defects', hue='Birth_asphyxia', data=train_gene )
fig3  

In [32]:
# Yes, Birth_asphyxia (lack of Oxygen) led to Birth_defects(singular is higher than multiple).

##### How does the Patient_Age vary across different levels of Genetic_Disorder?

In [ ]:
# Patient Age boxes according to each Disorder_Subclass :
plot = sns.boxplot( y='Disorder_Subclass', x='Patient_Age', data=train_gene)
plot

In [34]:
# Patient Ages are of approximate range across all Disorder Subclasses.

#### Did Following up the treatment of the patients affect their lives?
#####          (I mean the parents carefullness) 

In [ ]:
status = train_gene.groupby("Follow-up")['Status'].value_counts().reset_index()
status

In [ ]:
%pip install nbformat>=4.2.0

In [ ]:
fig_status = px.sunburst(status, path=['Follow-up', 'Status'], values='count') 
fig_status.show()

In [38]:
# Low Follow up led to more deceased Patients.

##### Is there any Birth defects besides Folic acid ?

In [ ]:
tree = train_gene[train_gene['Folic_acid'] == 'Yes'].groupby('Genetic_Disorder')['Birth_defects'].value_counts().reset_index()
fig = px.icicle(tree, path=['Genetic_Disorder', 'Birth_defects'], values='count')
fig.update_layout(
    iciclecolorway = ["pink", "lightgray", "grey"],
    margin = dict(t=50, l=25, r=25, b=25))

In [40]:
# Yes, although the mothers took Folic Acid but there were high Birth_defects(specially singular).

##### Does Maternal illness affect the abortion ?

In [ ]:
train_gene['Previous_abortion'] = train_gene['Previous_abortion'].replace(-99 , train_gene['Previous_abortion'].median())
train_gene.groupby('serious_maternal_illness')['Previous_abortion'].median()

In [42]:
# I replaced the -99 with nan in test data, it will be filled later by imputer:
test_gene['Previous_abortion'] = test_gene['Previous_abortion'].replace(-99 , 'nan')

In [43]:
# No, There is no relation between the maternal illness and the number of abortions.

##### What are the Ranges of Red Blood Cells & White Blood Cells ?

In [ ]:
pivot = pd.pivot_table(train_gene , values = ['Blood_cell_count','White_Blood_cell_count_thousand_per_microliter'], index = 'Genetic_Disorder' , aggfunc=np.median)
pivot

In [45]:
# White blood cells(WBC)   ----->	4500-11,000/mm3
# Red blood cells(RBC)     ----->   Male: 4.3-5.9 million/mm3 
#                          ----->   Female: 3.5-5.5 million/mm3
#So, they are normal.

##### Do babies Who Suferred Asphyxia (Alive), Are Still Suffereing Upnoramal Respiratory Rates?

In [ ]:
respiro = train_gene[(train_gene['Birth_asphyxia'] == 'Yes') & (train_gene['Status']=='Alive')]['Respiratory_Rate'].value_counts(normalize=True).reset_index()
respiro

In [ ]:
px.pie(respiro, names='Respiratory_Rate',values='proportion')

In [48]:
#No, Most of them are of normal respiratory rate.

### Detecting Imbalance :

In [ ]:
plt.figure(figsize=(8,6))
plt.subplot(1,2,1)
sns.countplot(train_gene,x='Genetic_Disorder')
plt.xticks(ticks=[0, 1, 2], labels=['Mitochondrial', 'Single-gene' , 'Multifactorial'])
plt.title("Genetic Disorder")
plt.subplot(1,2,2)
plt.pie(x=[train_gene['Genetic_Disorder'].value_counts()[0],train_gene['Genetic_Disorder'].value_counts()[1],train_gene['Genetic_Disorder'].value_counts()[2]],
        explode=[0.04,0.04,0.04],labels=['Mitochondrial','Multifactorial','Single-gene'],shadow=True,autopct='%.1f%%')
plt.title("Checking Imbalance")

# $${\color{orange}Machine-Learning}$$

###  Data Preparation
#### Splitting :

In [50]:
# Splitting The Train into x, y :
x_train = train_gene.drop('Genetic_Disorder', axis=1)
y_train = train_gene['Genetic_Disorder']
x_test  = test_gene

### Pipeline 

#### Imputing / Normalization / Encoding /SMOTE (Imbalance) :

In [51]:
# Listing the Categorical & Numerical columns :
categoric_columns = x_test.select_dtypes(include = 'O').columns.to_list()
numeric_columns = x_test.select_dtypes(include=['int' , 'float']).columns.tolist()

In [ ]:
%pip install scikit-learn

In [ ]:
%pip install imblearn

In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
# Define the preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),  # Impute numerical columns with median
            ('scaler', RobustScaler())  # Scale numerical columns
        ]), numeric_columns),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),  # Impute categorical columns with mode
            ('encoder', OneHotEncoder(handle_unknown='ignore'))  # Encode categorical columns
        ]), categoric_columns)
    ]
)
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),  # Apply preprocessing
    ('smote', SMOTE(sampling_strategy = 'auto', random_state=42)),  # Apply balancing
])

In [55]:
# Encoding the Target Column
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

In [56]:
# Fit and resample the training data
x_train_transformed, y_train_transformed = pipeline.fit_resample(x_train, y_train_encoded)
# For the test data, only apply the preprocessing (no resampling)
x_test_transformed = pipeline.named_steps['preprocessor'].transform(x_test)

In [ ]:
# Check the class distribution after SMOTE
class_distribution = pd.Series(y_train_transformed).value_counts()
print("Class distribution after SMOTE:")
print(class_distribution)

###  Model Selesction:
#### Hyperparameter Tuning / Grid Search / Randomized Search / Confusion Matrix / F1 Score :

### $${\color{orange}Model-Selection}$$

### Logistic Regression:

#### Grid Search with Cross Validation :

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Step 1: Define the logistic regression model
lr = LogisticRegression(random_state=100)

# Step 2: Define the hyperparameter grid for GridSearchCV
param_grid = {'C': [0.0001, 0.01, 0.1], 'penalty': ['l1', 'l2'], 'solver': ['liblinear', 'saga'] ,'class_weight': [{0: 1, 1: 1, 2: 2}, 'balanced']}

# Step 3: Perform Grid Search with Cross-Validation
grid_search = GridSearchCV(lr, param_grid, cv=5, scoring='accuracy')
grid_search.fit(x_train_transformed, y_train_transformed)

# Step 4: Extract the best model and its parameters
best_model = grid_search.best_estimator_

print(f"The best score: {grid_search.best_score_}, with the best parameters: {grid_search.best_params_}")

#### Cross Val Score On The Best Model (Grid Search) :

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from sklearn.metrics import classification_report
#Perform Cross val score with the best model (using F1 micro as the scoring metric)
best_model = grid_search.best_estimator_
cv_scores1 = cross_val_score(best_model, x_train_transformed, y_train_transformed, cv=5, scoring='f1_micro')
print(f'Cross-Validation Scores with Best Model: {cv_scores1}')
print(f'Mean Cross-Validation F1 Score: {cv_scores1.mean():.4f}')

### Decision Tree :

#### RandomizedSearch:

In [ ]:
from scipy.stats import randint
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
# Setup the parameters and distributions to sample from: param_dist
param_dist = {"max_depth": [3, 5],
              "max_features": randint(1, 9),
              "min_samples_leaf": randint(1, 9),
              "criterion": ["gini", "entropy"],
              "class_weight": [None, 'balanced'] }
# Instantiate a Decision Tree classifier: tree
tree = DecisionTreeClassifier(random_state=100)
# Instantiate the RandomizedSearchCV object: tree_cv
tree_cv = RandomizedSearchCV(tree, param_dist, cv=5 , random_state = 100)
# Fit it to the data
tree_cv.fit(x_train_transformed , y_train_transformed)
# Print the tuned parameters and score
print("Tuned Decision Tree Parameters: {}".format(tree_cv.best_params_))
print("Best score is {}".format(tree_cv.best_score_))

#### Cross Val Score On The Best Model (Randomized Search)

In [ ]:
from sklearn.model_selection import cross_val_predict, cross_val_score
#Initialize the best model from RandomizedSearchCV
best_tree = tree_cv.best_estimator_
# Perform cross-validation with f1_macro scoring
cv_scores3 = cross_val_score(best_tree, x_train_transformed, y_train_transformed, cv=5, scoring='f1_micro')
# Print cross-validation scores
print(f'Cross-Validation Scores (f1_macro): {cv_scores3}')
print(f'Mean Cross-Validation f1_macro Score: {cv_scores3.mean():.4f}')

### RANDOM FOREST 

#### RandomizedSearch Using Stratified Cross Val Score:

In [ ]:
from scipy.stats import randint
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import StratifiedKFold
# Setup the parameters and distributions to sample from: 
param_dist = {"n_estimators":[100,200,300],
              "max_depth": randint(1,9),
              "max_features": randint(1, 9),
              "min_samples_leaf": randint(1, 9),
              "criterion": ["gini", "entropy"],
              "class_weight": [None, 'balanced'] }
# Instantiate a Random Forest classifier: 
forest = RandomForestClassifier(random_state=100)
# Instantiate the RandomizedSearchCV object: 
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
forest_cv = RandomizedSearchCV(forest, param_dist, cv=stratified_kfold , random_state=100)
# Fit it to the data
forest_cv.fit(x_train_transformed,y_train_transformed)
# Print the tuned parameters and score
print("Tuned Decision Tree Parameters: {}".format(forest_cv.best_params_))
print("Best score is {}".format(forest_cv.best_score_))

#### Cross Val Score On The Best Estimator :

In [ ]:
from sklearn.model_selection import cross_val_score
#Initialize the best model from RandomizedSearchCV
best_forest = forest_cv.best_estimator_
# Perform cross-validation with f1_macro scoring
cv_scores4 = cross_val_score(best_forest, x_train_transformed, y_train_transformed, cv=stratified_kfold, scoring='f1_micro')
# Print cross-validation scores
print(f'Cross-Validation Scores (f1_macro): {cv_scores4}')
print(f'Mean Cross-Validation f1_macro Score: {cv_scores4.mean():.4f}')

### XGBOOST CLASSIFIER 

#### StratifiedKFold Cross Val Score :

In [64]:
# %pip install xgboost

In [ ]:
# %pip install xgboost==1.5.0
%pip install scikit-learn==1.0

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold , cross_val_score
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    booster='gbtree',              
    learning_rate=0.2,             
    n_estimators=200,               
    max_depth=4,                   
    min_child_weight=5,                     
    gamma=0.001,                                              
    alpha=0.6,                          
    eval_metric='mlogloss', 
    objective='multi:softprob',
    num_class=3,     
    random_state=42                 
)
# Set up StratifiedKFold with 5 folds
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Perform cross-validation
cv_scores5 = cross_val_score(xgb_model, x_train_transformed, y_train_transformed, cv=stratified_kfold, scoring='f1_micro')
# Print the results
print("Cross-Validation Scores: ", cv_scores5)
print("Mean Cross-Validation Score: ", np.mean(cv_scores5))

In [ ]:
# # Perform cross-validation predictions
y_pred_xgt = cross_val_predict(xgb_model, x_train_transformed, y_train_transformed, cv=5)
# Print classification report
print("Classification Report:")
print(classification_report(y_train_transformed, y_pred_xgt))

#### StratifiedKFold Cross Val Score / Class Weight 
#### Giving Weight to class 3 ---> increase the recall

In [69]:
def cross_val_with_sample_weights(X, y, model, class_weights, cv_folds=5):
    skf = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=42)
    f1_scores = []

    for fold, (train_index, val_index) in enumerate(skf.split(X, y), start=1):
        X_train, X_val = X[train_index], X[val_index]
        y_train, y_val = y[train_index], y[val_index]

        # Compute sample weights for the current fold
        sample_weights = compute_sample_weights(y_train, class_weights)

        # Fit the model with sample weights
        model.fit(X_train, y_train, sample_weight=sample_weights)

        # Predict and evaluate
        y_pred = model.predict(X_val)

        # Compute F1 micro score
        f1_micro = f1_score(y_val, y_pred, average='micro')
        f1_scores.append(f1_micro)

        # Print classification report for this fold
        report = classification_report(y_val, y_pred, zero_division=1, target_names=['Class 0', 'Class 1', 'Class 2'])
        print(f"Fold {fold} Classification Report:")
        print(report)
        print(f"Fold {fold} F1 Micro Score: {f1_micro:.4f}")
        print("-" * 50)
    return f1_scores

def compute_sample_weights(y, class_weights):
    weights = np.zeros_like(y, dtype=float)
    for class_label, weight in class_weights.items():
        weights[y == class_label] = weight
    return weights


In [70]:
class_weights = {0: 1.5, 1: 1.5, 2: 2}
# Initialize the model
model = XGBClassifier(
    learning_rate=0.2,
    gamma=0.001,                                              
    alpha=0.5, 
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    use_label_encoder=False
)

# Ensure X and y are properly loaded as NumPy arrays
x_train_transformed = np.array(x_train_transformed)
y_train_transformed = np.array(y_train_transformed)

In [ ]:
f1_scores = cross_val_with_sample_weights(x_train_transformed,y_train_transformed, model, class_weights, cv_folds=5)
print(f"Cross-validation F1 micro scores: {f1_scores}")
print(f"Mean F1 micro score: {np.mean(f1_scores):.4f}")

### CatBoost Classifier 

#### StratifiedKFold Cross Val Score :

In [ ]:
%pip install catboost --user

In [ ]:
from sklearn.model_selection import cross_val_score
from catboost import CatBoostClassifier
catboost_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.2,
    depth=4,
    loss_function='MultiClass',
    eval_metric='TotalF1',
    random_state=42
)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores6 = cross_val_score(catboost_model, x_train_transformed, y_train_transformed, cv=stratified_kfold, scoring='f1_weighted')
# Print the results
print("Cross-Validation Scores: ", cv_scores6)
print("Mean Cross-Validation Score: ", np.mean(cv_scores6))

### Lightgbm :

#### StratifiedKFold Cross Val Score

In [ ]:
%pip install lightgbm

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import cross_val_score
# Initialize the LightGBM classifier
light= lgb.LGBMClassifier(
    objective='multiclass',
    num_class=3,
    boosting_type='gbdt',
    metric='multi_logloss',
    num_leaves=31,
    learning_rate=0.05,
    feature_fraction=0.9
)
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# Perform cross-validation
cv_scores7 = cross_val_score(light, x_train_transformed, y_train_transformed, cv=stratified_kfold, scoring='f1_micro')

print(f'Cross-Validation F1 Micro Scores: {cv_scores7}')
print(f'Mean Cross-Validation F1 Micro Score: {np.mean(cv_scores7):.4f}')

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report 
# Use cross_val_predict to get predictions
y_pred_light = cross_val_predict(light, x_train_transformed, y_train_transformed, cv=5, method='predict')
# Generate the classification report
report = classification_report(y_train_transformed, y_pred_light, target_names=['Class 0', 'Class 1', 'Class 2'])
# Display the report
print("Classification Report:")
print(report)


In [ ]:
# Plot the confusion matrix
from sklearn.metrics import  confusion_matrix
cm = confusion_matrix(y_train_transformed, y_pred_light)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Class 1', 'Class 2', 'Class 3'], yticklabels=['Class 1', 'Class 2', 'Class 3'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()


### $${\color{orange}Comparing-Models}$$

In [ ]:
# Create a DataFrame from the scores
df_cv_scores = pd.DataFrame({
    'Model': ['Logistic', 'Decision Tree', 'Random Forest' , 'XGBoost','CatBoost' , 'XGBoost(Weight)', 'LightGBM'],
    'Mean_F1_Score': [pd.Series(scores).mean() for scores in [cv_scores1, cv_scores3, cv_scores4 , cv_scores5, cv_scores6 , f1_scores , cv_scores7]]
})
print(df_cv_scores)

In [ ]:
# Plot bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x='Model', y='Mean_F1_Score', data=df_cv_scores, palette='deep')
plt.title('Comparison of Cross Val Scores for Different Models')
plt.xlabel('Model')
plt.ylabel('Mean F1 Score')
plt.grid(True)
plt.show()

### Fitting The XGBoost (Weight) :

In [ ]:
model.fit(x_train_transformed, y_train_transformed , sample_weight=compute_sample_weights(y_train_transformed, class_weights))

### $${\color{orange}Feature-Selection}$$

### Using Xgboost :

In [ ]:
# Get feature importances from your trained model
feature_imp = model.feature_importances_
# Create a Series with feature importances
feature_imp_frame = pd.Series(feature_imp, index=pipeline.named_steps['preprocessor'].get_feature_names_out()).sort_values(ascending=False)
# Reset the index to create a DataFrame
feature_imp_frame = feature_imp_frame.reset_index()
feature_imp_frame.columns = ['Feature', 'Importance']  # Rename columns
# Exclude the first 9 rows and select the next 10 features (rows 10 to 19)
top_features = feature_imp_frame.iloc[9:19]
# Plot the selected features using a pie chart
fig = px.pie(top_features, names='Feature', values='Importance', title='Feature Importance')
# Show the plot
fig.show()

#### XGBOOST ON TEST DATA :

In [220]:
y_last_pred = model.predict(x_test_transformed)

In [221]:
cleaned_gene = train_gene.to_csv('cleaned_gene.csv', index=False)

### Saving The Model :

In [222]:
import pickle
# Save the model
with open('genes_model.pkl', 'wb') as file:
    pickle.dump(model, file)

In [236]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler, OneHotEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from keras.models import Sequential
from keras.layers import Dense, Dropout


In [246]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE

# Load the data
train_gene = pd.read_csv(r"./train.csv")
test_gene = pd.read_csv(r"./test.csv")

# Strip any whitespace from column names
train_gene.columns = train_gene.columns.str.strip()
test_gene.columns = test_gene.columns.str.strip()

# Rename the 'Genetic Disorder' column for easier access
train_gene.rename(columns={'Genetic Disorder': 'Genetic_Disorder'}, inplace=True)

# Drop 'Disorder Subclass' from training data if it's not in the test data
if 'Disorder Subclass' in train_gene.columns and 'Disorder Subclass' not in test_gene.columns:
    train_gene = train_gene.drop('Disorder Subclass', axis=1)

# Splitting the dataset into features and target
x_train = train_gene.drop('Genetic_Disorder', axis=1)  # Features
y_train = train_gene['Genetic_Disorder']  # Target variable
x_test = test_gene  # Test features (no target variable)

# Encode the target variable
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Identify numeric and categorical columns
categoric_columns = x_train.select_dtypes(include='O').columns.to_list()
numeric_columns = x_train.select_dtypes(include=['int', 'float']).columns.tolist()

# Define the preprocessing steps
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), numeric_columns),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore'))
        ]), categoric_columns)
    ]
)

# Step 1: Fit and transform the training data using the preprocessing pipeline
x_train_transformed = preprocessor.fit_transform(x_train)

# Step 2: Apply SMOTE on the transformed training data
smote = SMOTE(sampling_strategy='auto', random_state=42)
x_train_resampled, y_train_resampled = smote.fit_resample(x_train_transformed, y_train_encoded)

# Prepare the test data (no SMOTE)
x_test_transformed = preprocessor.transform(x_test)

# You can now use x_train_resampled, y_train_resampled for training your model
# and x_test_transformed for making predictions.


# DEEP LEARNING

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Build the deep learning model
model = Sequential()
model.add(Dense(128, activation='relu', input_shape=(x_train_resampled.shape[1],)))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(len(label_encoder.classes_), activation='softmax'))  # Number of classes for output

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Early stopping to prevent overfitting
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Train the model
history = model.fit(x_train_resampled, y_train_resampled, 
                    validation_split=0.2, 
                    epochs=50, 
                    batch_size=32, 
                    verbose=1,
                    callbacks=[early_stopping])

# Make predictions on the test set
y_pred = model.predict(x_test_transformed)
y_pred_classes = np.argmax(y_pred, axis=1)  # Convert probabilities to class labels

# Map the encoded predictions back to original class names
predicted_classes = label_encoder.inverse_transform(y_pred_classes)

# Plot training & validation accuracy values
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.models import load_model

# Save the model after training
model.save("genetic_disorder_model.h5")

# Load the model for testing or further predictions
loaded_model = load_model("genetic_disorder_model.h5")

# Use the loaded model to make predictions on test data
y_pred = loaded_model.predict(x_test_transformed)
y_pred_classes = np.argmax(y_pred, axis=1)  # Convert probabilities to class labels

# Map the encoded predictions back to original class names
predicted_classes = label_encoder.inverse_transform(y_pred_classes)


In [263]:
import joblib

In [ ]:
# Save the label encoder for future use
joblib.dump(label_encoder, "label_encoder.pkl")